In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

In [5]:
list(Path("datasets").rglob("*.csv"))

[WindowsPath('datasets/EducationLevel.csv'),
 WindowsPath('datasets/Employee.csv'),
 WindowsPath('datasets/PerformanceRating.csv'),
 WindowsPath('datasets/RatingLevel.csv'),
 WindowsPath('datasets/SatisfiedLevel.csv')]

In [6]:
df_employees = pd.read_csv("datasets/Employee.csv")
df_reviews = pd.read_csv("datasets/PerformanceRating.csv")

In [7]:
print("Employees:", df_employees.shape)

Employees: (1470, 23)


In [9]:
print("Employees columns:")
print(df_employees.columns.tolist())

Employees columns:
['EmployeeID', 'FirstName', 'LastName', 'Gender', 'Age', 'BusinessTravel', 'Department', 'DistanceFromHome (KM)', 'State', 'Ethnicity', 'Education', 'EducationField', 'JobRole', 'MaritalStatus', 'Salary', 'StockOptionLevel', 'OverTime', 'HireDate', 'Attrition', 'YearsAtCompany', 'YearsInMostRecentRole', 'YearsSinceLastPromotion', 'YearsWithCurrManager']


In [8]:
print("Reviews:", df_reviews.shape)

Reviews: (6709, 11)


In [10]:
print("\nReviews columns:")
print(df_reviews.columns.tolist())


Reviews columns:
['PerformanceID', 'EmployeeID', 'ReviewDate', 'EnvironmentSatisfaction', 'JobSatisfaction', 'RelationshipSatisfaction', 'TrainingOpportunitiesWithinYear', 'TrainingOpportunitiesTaken', 'WorkLifeBalance', 'SelfRating', 'ManagerRating']


In [11]:
df_employees.head()

,EmployeeID,FirstName,LastName,Gender,Age,BusinessTravel,Department,DistanceFromHome (KM),State,Ethnicity,Education,EducationField,JobRole,MaritalStatus,Salary,StockOptionLevel,OverTime,HireDate,Attrition,YearsAtCompany,YearsInMostRecentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,3012-1A41,Leonelle,Simco,Female,30,Some Travel,Sales,27,IL,White,5,Marketing,Sales Executive,Divorced,102059,1,No,2012-01-03,No,10,4,9,7
1,CBCB-9C9D,Leonerd,Aland,Male,38,Some Travel,Sales,23,CA,White,4,Marketing,Sales Executive,Single,157718,0,Yes,2012-01-04,No,10,6,10,0
2,95D7-1CE9,Ahmed,Sykes,Male,43,Some Travel,Human Resources,29,CA,Asian or Asian American,4,Marketing,HR Business Partner,Married,309964,1,No,2012-01-04,No,10,6,10,8
3,47A0-559B,Ermentrude,Berrie,Non-Binary,39,Some Travel,Technology,12,IL,White,3,Computer Science,Engineering Manager,Married,293132,0,No,2012-01-05,No,10,10,10,0
4,42CC-040A,Stace,Savege,Female,29,Some Travel,Human Resources,29,CA,White,2,Technical Degree,Recruiter,Single,49606,0,No,2012-01-05,Yes,6,1,1,6


In [14]:
df_workforce = df_employees.copy()

In [15]:
# Binary target variable for logistic regression
df_workforce["AttritionBinary"] = (df_workforce["Attrition"].map({"No": 0, "Yes": 1}))

In [16]:
print("Employees:", len(df_workforce))
print("Unique employees:", df_workforce["EmployeeID"].nunique())
print("Leavers:", df_workforce["AttritionBinary"].sum())
print("Missing target values:", df_workforce["AttritionBinary"].isna().sum())

Employees: 1470
Unique employees: 1470
Leavers: 237
Missing target values: 0


In [18]:
# Adding the hire date to the ratings table
df_reviews_merged = df_reviews.merge(
    df_workforce[["EmployeeID", "HireDate"]],
    on="EmployeeID",
    how="left",
    validate="many_to_one"
)

In [21]:
# Converting dates and filtering records
df_reviews_merged["ReviewDate"] = pd.to_datetime(df_reviews_merged["ReviewDate"])
df_reviews_merged["HireDate"] = pd.to_datetime(df_reviews_merged["HireDate"])
df_eligible_reviews = df_reviews_merged[df_reviews_merged["ReviewDate"] >= df_reviews_merged["HireDate"]].copy()

In [23]:
# Selecting the most recent rating
df_latest_reviews = (
    df_eligible_reviews
    .sort_values("ReviewDate")
    .drop_duplicates(subset="EmployeeID", keep="last")
    .copy()
)

In [25]:
# checking
assert len(df_latest_reviews) == 1217
assert df_latest_reviews["EmployeeID"].is_unique

In [26]:
# Preparing the reviewed cohort
df_reviewed = df_workforce.merge(
    df_latest_reviews.drop(columns="HireDate"),
    on="EmployeeID",
    how="inner",
    validate="one_to_one"
)

In [28]:
# Saving processed datasets
processed_dir = Path("datasets/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

df_workforce.to_csv(processed_dir / "workforce_analysis.csv", index=False)
df_reviewed.to_csv(processed_dir / "reviewed_analysis.csv", index=False)